# Gemma 4 E4B 뉴스 분석 LoRA 파인튜닝 Colab 노트북

이 노트북은 로컬 Ollama 비교에서 1위로 선정된 `gemma4:e4b` 계열을 프로젝트 데이터셋으로 파인튜닝하기 위한 Colab 실행 템플릿입니다.

중요한 전제:
- Ollama의 `gemma4:e4b`는 Q4_K_M GGUF 실행용 모델이므로 Colab에서 직접 역전파 학습하지 않습니다.
- Colab에서는 Hugging Face 원본 모델 `google/gemma-4-E4B-it`를 4-bit QLoRA로 학습합니다.
- 학습 결과는 LoRA adapter로 저장하고, 필요하면 GGUF `q4_k_m`로 내보내 로컬 Ollama에 다시 등록합니다.
- 모든 작업 경로는 Google Drive의 `/MyDrive/dev-jinro`로 고정되어 있습니다.
  - 학습 데이터: `/MyDrive/dev-jinro/data/chatml_dataset.jsonl`
  - 학습 결과: `/MyDrive/dev-jinro/outputs/gemma4_news_analyzer`
  - 압축 결과: `/MyDrive/dev-jinro/gemma4_news_analyzer_outputs.zip`

권장 런타임:
- Colab Pro: L4 또는 A100 GPU
- 무료 Colab: T4에서도 `MAX_STEPS`, `MAX_SEQ_LENGTH`, batch size를 줄이면 시도 가능


## 1. 패키지 설치

Transformers + TRL + PEFT + BitsAndBytes 기반으로 QLoRA 학습을 수행합니다. Colab 런타임을 새로 켤 때마다 한 번 실행하세요.

In [ ]:
!pip -q install -U trl peft accelerate bitsandbytes datasets transformers huggingface_hub


## 2. Drive 작업공간, 인증 및 설정

모든 데이터와 결과는 Google Drive의 `/MyDrive/dev-jinro` 아래에서 처리합니다. Hugging Face 모델 접근에 토큰이 필요한 경우 Colab 좌측 `Secrets`에 `HF_TOKEN`을 저장하거나, 아래 `notebook_login()` 주석을 해제하세요.

In [ ]:
import json
import os
import re
import shutil
import textwrap
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login, notebook_login
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTConfig, SFTTrainer

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/dev-jinro")
DATA_DIR = DRIVE_ROOT / "data"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "gemma4_news_analyzer"
ADAPTER_DIR = OUTPUT_ROOT / "lora_adapter"
MERGED_DIR = OUTPUT_ROOT / "merged_16bit"
GGUF_DIR = OUTPUT_ROOT / "gguf_q4_k_m"

for directory in [DRIVE_ROOT, DATA_DIR, OUTPUT_ROOT, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    HF_TOKEN = userdata.get("HF_TOKEN") or os.getenv("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("HF_TOKEN이 없습니다. 모델 접근이 막히면 notebook_login() 주석을 해제해 로그인하세요.")
    # notebook_login()

BASE_MODEL_ID = "google/gemma-4-E4B-it"
DATASET_PATH = DATA_DIR / "chatml_dataset.jsonl"

MAX_SEQ_LENGTH = 768
MAX_STEPS = 60
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
SEED = 3407

print("Drive workspace:", DRIVE_ROOT)
print("Dataset path:", DATASET_PATH)
print("Output root:", OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Base model:", BASE_MODEL_ID)


## 3. 학습 데이터 준비

프로젝트의 `ai-engine/data/chatml_dataset.jsonl` 파일을 Drive의 `dev-jinro/data/chatml_dataset.jsonl`에 둡니다. 이미 Drive에 파일을 넣었다면 이 셀은 업로드를 건너뜁니다.

In [ ]:
from google.colab import files

if not DATASET_PATH.exists():
    print("Drive에 학습 데이터가 없습니다. chatml_dataset.jsonl 파일을 업로드하세요.")
    uploaded = files.upload()
    if "chatml_dataset.jsonl" not in uploaded:
        raise FileNotFoundError("업로드한 파일 중 chatml_dataset.jsonl이 없습니다.")
    DATASET_PATH.write_bytes(uploaded["chatml_dataset.jsonl"])
    print("Uploaded dataset saved to Drive:", DATASET_PATH)
else:
    print("Using existing Drive dataset:", DATASET_PATH)

print("Dataset path:", DATASET_PATH)
print("Dataset size:", DATASET_PATH.stat().st_size, "bytes")


## 4. 데이터 검증

각 줄이 `{ "messages": [...] }` 형식인지 확인하고, assistant 응답이 JSON으로 파싱되는 비율을 확인합니다.

In [ ]:
def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL {line_no}번째 줄 파싱 실패: {exc}") from exc
    return rows

def extract_last_assistant(messages):
    for message in reversed(messages):
        if message.get("role") == "assistant":
            return message.get("content", "")
    return ""

rows = read_jsonl(DATASET_PATH)
if not rows:
    raise ValueError("데이터셋이 비어 있습니다.")

invalid = []
assistant_json_ok = 0
for index, row in enumerate(rows):
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) < 2:
        invalid.append(index)
        continue
    if not all("role" in message and "content" in message for message in messages):
        invalid.append(index)
        continue
    assistant_text = extract_last_assistant(messages)
    try:
        json.loads(assistant_text)
        assistant_json_ok += 1
    except Exception:
        pass

print("rows:", len(rows))
print("invalid rows:", len(invalid))
print("assistant JSON rate:", f"{assistant_json_ok / len(rows) * 100:.1f}%")
print("sample:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2)[:1200])

if invalid:
    raise ValueError(f"messages 형식이 잘못된 row가 있습니다. 예: {invalid[:10]}")


## 5. 데이터셋 로드 및 train/eval 분리

In [ ]:
dataset = load_dataset("json", data_files=str(DATASET_PATH), split="train")
test_size = 0.1 if len(dataset) >= 20 else 0.2
split_dataset = dataset.train_test_split(test_size=test_size, seed=SEED)

print(split_dataset)


## 6. Gemma 4 E4B 모델 로드 및 LoRA 설정

공식 HF 모델을 4-bit로 로드하고 LoRA adapter를 붙입니다. GPU 메모리가 부족하면 `MAX_SEQ_LENGTH`, `MAX_STEPS`, `BATCH_SIZE`를 낮추세요.

In [ ]:
api = HfApi()
try:
    api.model_info(BASE_MODEL_ID, token=HF_TOKEN)
    print("HF model metadata check: OK")
except Exception as exc:
    print("HF model metadata check failed. 모델 라이선스 동의 또는 HF_TOKEN이 필요할 수 있습니다.")
    print(type(exc).__name__, str(exc)[:500])

MODEL_LOAD_BACKEND = "transformers_peft"
ACTIVE_MODEL_ID = BASE_MODEL_ID

print("Loading Gemma 4 E4B with native Transformers + PEFT QLoRA.")
print("중요: 이전 셀에서 Unsloth를 import한 런타임이면 런타임을 재시작한 뒤 이 노트북을 처음부터 실행하세요.")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    device_map="auto",
    quantization_config=quantization_config,
    attn_implementation="sdpa",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
try:
    model.enable_input_require_grads()
except AttributeError:
    pass

# Gemma 4는 q_proj 등이 Gemma4ClippableLinear 래퍼입니다.
# PEFT는 이 래퍼를 직접 지원하지 않으므로 내부 Linear4bit 서브모듈을 target으로 지정합니다.
LORA_TARGET_MODULES = [
    "q_proj.linear",
    "k_proj.linear",
    "v_proj.linear",
    "o_proj.linear",
    "gate_proj.linear",
    "up_proj.linear",
    "down_proj.linear",
]
matched_lora_targets = sorted(
    name for name, _module in model.named_modules()
    if any(name.endswith(target) for target in LORA_TARGET_MODULES)
)
print("Matched LoRA target module count:", len(matched_lora_targets))
print("Matched LoRA target samples:", matched_lora_targets[:8])
if not matched_lora_targets:
    raise ValueError("Gemma 4 내부 linear LoRA target을 찾지 못했습니다. Transformers 모델 구조를 확인하세요.")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

print("Model backend:", MODEL_LOAD_BACKEND)
print("Active model:", ACTIVE_MODEL_ID)
print("LoRA model prepared.")


## 7. Chat template 적용

`messages` 배열을 Gemma 4 채팅 템플릿의 단일 학습 문자열로 변환합니다.

In [ ]:
def apply_template(example):
    try:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    return {"text": text}

train_dataset = split_dataset["train"].map(apply_template, remove_columns=split_dataset["train"].column_names)
eval_dataset = split_dataset["test"].map(apply_template, remove_columns=split_dataset["test"].column_names)

print(train_dataset)
print(train_dataset[0]["text"][:1200])


## 8. SFTTrainer 구성

TRL 버전에 따라 `max_seq_length`와 `max_length`, `processing_class`와 `tokenizer` 인자가 다를 수 있어 호환 래퍼를 둡니다.

In [ ]:
def make_sft_config():
    common = dict(
        output_dir=str(OUTPUT_ROOT / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        packing=False,
        report_to="none",
        save_steps=MAX_STEPS,
        do_eval=False,
        per_device_eval_batch_size=1,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
    )
    last_error = None
    eval_key_candidates = [{"eval_strategy": "no"}, {"evaluation_strategy": "no"}, {}]
    for eval_kwargs in eval_key_candidates:
        for length_key in ["max_seq_length", "max_length"]:
            try:
                return SFTConfig(**common, **eval_kwargs, **{length_key: MAX_SEQ_LENGTH})
            except TypeError as exc:
                last_error = exc
    raise last_error

sft_config = make_sft_config()

def build_trainer():
    base_kwargs = dict(
        model=model,
        train_dataset=train_dataset,
        args=sft_config,
    )
    attempts = [
        {"processing_class": tokenizer, "dataset_text_field": "text"},
        {"tokenizer": tokenizer, "dataset_text_field": "text"},
        {"processing_class": tokenizer},
        {"tokenizer": tokenizer},
    ]
    last_error = None
    for extra_kwargs in attempts:
        try:
            return SFTTrainer(**base_kwargs, **extra_kwargs)
        except TypeError as exc:
            last_error = exc
    raise last_error

trainer = build_trainer()
trainer


## 9. 학습 실행 및 adapter 저장

In [ ]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"CUDA memory before train: free={free_mem / 1024**3:.2f}GB / total={total_mem / 1024**3:.2f}GB")

try:
    train_result = trainer.train()
except RuntimeError as exc:
    message = str(exc)
    if "CUBLAS_STATUS_ALLOC_FAILED" in message or "out of memory" in message.lower():
        print("GPU 메모리가 부족합니다. 런타임을 재시작한 뒤 MAX_SEQ_LENGTH를 512, MAX_STEPS를 30으로 낮춰 다시 실행하세요.")
    raise
print(train_result)

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("Saved LoRA adapter:", ADAPTER_DIR)
print("Adapter files:")
for path in sorted(ADAPTER_DIR.rglob("*")):
    if path.is_file():
        print(path)


## 10. 간단한 JSON 출력 검증

학습된 adapter가 프로젝트가 기대하는 JSON 스키마로 답하는지 간단히 확인합니다.

In [ ]:
model.eval()

test_messages = [
    {
        "role": "system",
        "content": "당신은 엄격하고 객관적인 한국어 뉴스 분석 AI입니다. sentiment_score, bias_score, factuality_score, summary만 포함한 JSON 객체 하나만 출력하세요.",
    },
    {
        "role": "user",
        "content": "[기사 제목]: 교통 혼잡 완화를 위한 버스 전용 차로 확대 검토\n[기사 본문]: 서울시는 출퇴근 시간 주요 간선도로의 혼잡을 줄이기 위해 버스 전용 차로 확대를 검토하고 있다. 시민단체는 대중교통 이용 확대를 기대했고 일부 상인은 접근성 저하를 우려했다.",
    },
]

try:
    prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError:
    prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)

generation_device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer(prompt, return_tensors="pt").to(generation_device)
outputs = model.generate(
    **inputs,
    max_new_tokens=220,
    temperature=0.0,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)
decoded = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print(decoded)

match = re.search(r"\{.*\}", decoded, flags=re.S)
if match:
    parsed = json.loads(match.group(0))
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
else:
    print("JSON 객체를 찾지 못했습니다. MAX_STEPS를 늘리거나 데이터 품질을 확인하세요.")


## 11. GGUF q4_k_m export 및 Ollama Modelfile 생성

Transformers/PEFT 경로에서는 LoRA adapter를 우선 산출물로 사용합니다. GGUF가 필요하면 adapter를 merge한 뒤 별도 변환 노트북 또는 llama.cpp에서 변환합니다.

In [ ]:
export_note = OUTPUT_ROOT / "GGUF_EXPORT_NOTE.md"
export_note.write_text(
    "# GGUF export 안내\n\n"
    "현재 학습은 Transformers/PEFT QLoRA backend로 진행되어 LoRA adapter가 저장되었습니다.\n"
    "GGUF가 필요하면 16bit base model에 adapter를 merge한 뒤 llama.cpp에서 q4_k_m으로 변환하세요.\n"
    "이 노트북에서는 Unsloth 패치 충돌을 피하기 위해 GGUF 자동 export를 수행하지 않습니다.\n"
    f"\n- Base model: {ACTIVE_MODEL_ID}\n"
    f"- Adapter: {ADAPTER_DIR}\n",
    encoding="utf-8",
)
print("GGUF export note:", export_note)

gguf_files = sorted(GGUF_DIR.glob("*.gguf"))
gguf_from_path = f"./{gguf_files[0].relative_to(OUTPUT_ROOT)}" if gguf_files else "./gguf_q4_k_m/gemma4_news_analyzer-q4_k_m.gguf"

modelfile_path = OUTPUT_ROOT / "Modelfile"
modelfile_path.write_text(
    textwrap.dedent(
        f'''\
        FROM {gguf_from_path}
        TEMPLATE "{{ .Prompt }}"
        PARAMETER temperature 0
        PARAMETER top_p 0.9
        PARAMETER top_k 64
        '''
    ),
    encoding="utf-8",
)

print(modelfile_path.read_text())
print("Output files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path)


## 12. 결과 압축 및 다운로드

학습 결과는 이미 Drive의 `dev-jinro/outputs/gemma4_news_analyzer` 아래에 저장됩니다. 이 셀은 같은 Drive 폴더 안에 zip 파일을 만들고, 필요하면 로컬 다운로드도 수행합니다.

로컬 실행 예시:

```bash
ollama create gemma4-news-analyzer:q4_k_m -f Modelfile
ollama run gemma4-news-analyzer:q4_k_m
```

In [ ]:
archive_base = DRIVE_ROOT / "gemma4_news_analyzer_outputs"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(OUTPUT_ROOT))
print("Archive:", archive_path)

print("Drive workspace:", DRIVE_ROOT)
print("Model output directory:", OUTPUT_ROOT)

from google.colab import files
files.download(archive_path)
